# UK Jobs — Data Cleaning Pipeline

**Input :** `data/UK_data/uk_jobs_parsed.csv`
**Output:** `data/UK_data/uk_jobs_final.csv`

### Cleaning steps
| # | Step |
|---|------|
| 1 | Fix `job_id` → string (strip float `.0`) |
| 2 | Fix `posted_date` → `YYYY-MM-DD` |
| 3 | Set `country` column |
| 4 | Drop rows with missing `hard_skills` |
| 5 | Drop rows with missing `parsed_title` |
| 6 | Nullify zeros in salary columns (0 → NaN) |
| 7 | Strip whitespace from all text columns |
| 8 | Drop `job_category == 'other'` |
| 9 | Drop unused raw columns |

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

COUNTRY      = 'UK'
RAW_PARSED   = 'data/UK_data/uk_jobs_parsed.csv'
OUTPUT_CSV   = 'data/UK_data/uk_jobs_final.csv'

df_raw = pd.read_csv(RAW_PARSED, encoding='utf-8-sig')
print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print()
print('Job category breakdown (raw):')
print(df_raw['job_category'].value_counts().to_string())
df_raw.head(3)

In [ ]:
def empty_counts(df):
    is_empty = df.isnull() | df.apply(lambda col: col.astype(str).str.strip() == '')
    empty = is_empty.sum()
    pct   = (empty / len(df) * 100).round(1)
    return pd.DataFrame({'empty': empty, 'pct_%': pct}).sort_values('pct_%', ascending=False)

print('=== Empty / Null counts — RAW ===')
print(empty_counts(df_raw).to_string())

In [ ]:
df = df_raw.copy()
drop_log = {}

### Step 1 — Fix job_id

In [ ]:
def clean_job_id(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan'):
        return ''
    try:
        return str(int(float(str(val).strip())))
    except (ValueError, OverflowError):
        return str(val).strip()

df['job_id'] = df['job_id'].apply(clean_job_id)
print('Sample :', df['job_id'].head(5).tolist())
print('Still ends .0:', df['job_id'].str.endswith('.0').sum())

### Step 2 — Fix posted_date → YYYY-MM-DD

In [ ]:
# Handles both DD-MM-YY (US) and YYYY-MM-DD (others) automatically
parsed = pd.to_datetime(df['posted_date'], format='%d-%m-%y', errors='coerce')
iso_mask = parsed.isna() & df['posted_date'].notna()
if iso_mask.any():
    parsed[iso_mask] = pd.to_datetime(df.loc[iso_mask, 'posted_date'], format='%Y-%m-%d', errors='coerce')
df['posted_date'] = parsed.dt.strftime('%Y-%m-%d')

print(f'Invalid dates : {df["posted_date"].isna().sum()}')
print(f'Sample        : {df["posted_date"].head(3).tolist()}')

### Step 3 — Set country column

In [ ]:
df['country'] = COUNTRY
print('Country set to:', COUNTRY)

### Step 4 — Drop rows with missing hard_skills

In [ ]:
before = len(df)
mask = df['hard_skills'].isna() | (df['hard_skills'].astype(str).str.strip() == '')
dropped = df[mask]['job_category'].value_counts()
df = df[~mask].copy()
drop_log['4_no_hard_skills'] = before - len(df)
print(f'Dropped: {drop_log["4_no_hard_skills"]}  |  Remaining: {len(df):,}')
print('Dropped by category:', dropped.to_dict())

### Step 5 — Drop rows with missing parsed_title

In [ ]:
before = len(df)
mask = df['parsed_title'].isna() | (df['parsed_title'].astype(str).str.strip() == '')
df = df[~mask].copy()
drop_log['5_no_parsed_title'] = before - len(df)
print(f'Dropped: {drop_log["5_no_parsed_title"]}  |  Remaining: {len(df):,}')

### Step 6 — Nullify zeros in salary columns (0 → NaN)

In [ ]:
for col in ['salary_min_annual', 'salary_max_annual']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    n_zero = (df[col] == 0.0).sum()
    df.loc[df[col] == 0.0, col] = np.nan
    print(f'{col}: {n_zero} zeros converted to NaN')

### Step 7 — Strip whitespace from all text columns

In [ ]:
str_cols = df.select_dtypes(include='object').columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()
print(f'Stripped {len(str_cols)} string columns')

### Step 8 — Drop job_category == 'other'

In [ ]:
before = len(df)
df = df[df['job_category'].str.lower() != 'other'].copy()
drop_log['8_drop_other_cat'] = before - len(df)
print(f'Dropped: {drop_log["8_drop_other_cat"]}  |  Remaining: {len(df):,}')
print()
print('Category breakdown (final):')
print(df['job_category'].value_counts().to_string())

### Step 9 — Drop unused raw columns

In [ ]:
cols_to_drop = [c for c in ['salary_min_raw', 'salary_max_raw'] if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f'Dropped columns: {cols_to_drop}')

---
### Cleaning Summary

In [ ]:
print('=== CLEANING SUMMARY ===')
print(f'  Raw rows       : {len(df_raw):,}')
for step, n in drop_log.items():
    print(f'  {step:<30}: -{n}')
print(f'  Final rows     : {len(df):,}')
print(f'  Rows retained  : {100 * len(df) / len(df_raw):.1f}%')
print()
print('=== Empty value counts — CLEANED ===')
print(empty_counts(df).to_string())

---
### Export

In [ ]:
COL_ORDER = [
    'job_id', 'source', 'country', 'url',
    'job_title', 'parsed_title', 'company', 'location', 'posted_date',
    'seniority', 'min_years_exp', 'employment_type', 'remote_status',
    'salary_min_annual', 'salary_max_annual',
    'hard_skills', 'soft_skills', 'education', 'job_category',
]

cols = [c for c in COL_ORDER if c in df.columns]
df_out = df[cols].copy()
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Exported to: {OUTPUT_CSV}')
print(f'  Rows    : {len(df_out):,}')
print(f'  Columns : {list(df_out.columns)}')

---
### Verification

In [ ]:
dv = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig', dtype={'job_id': str})
checks = {
    'job_id — no .0 suffix'      : not dv['job_id'].astype(str).str.endswith('.0').any(),
    'job_category — no other'    : (dv['job_category'].str.lower() == 'other').sum() == 0,
    'hard_skills — none empty'   : (dv['hard_skills'].isna() | (dv['hard_skills'] == '')).sum() == 0,
    'parsed_title — none empty'  : (dv['parsed_title'].isna() | (dv['parsed_title'] == '')).sum() == 0,
    'salary — no zeros'          : (dv['salary_min_annual'] == 0).sum() == 0,
    'no salary_midpoint col'     : 'salary_midpoint' not in dv.columns,
    'no raw salary cols'         : 'salary_min_raw' not in dv.columns,
    'country column present'     : 'country' in dv.columns,
}
for k, v in checks.items():
    print(f'  {"OK" if v else "FAIL"} {k}')
print()
print('All checks passed!' if all(checks.values()) else 'WARNING: some checks failed')
print()
dv.head(3)